In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

catalog = "workspace"
gold_schema = "insurance_gold"

spark.sql(
f"""
CREATE SCHEMA IF NOT EXISTS
{catalog}.{gold_schema}
"""
)

payment_df = spark.table(
"workspace.insurance_silver.insurance_payments"
)

policy_df = spark.table(
"workspace.insurance_silver.policy_master"
)

customer_df = spark.table(
"workspace.insurance_silver.customer_master"
)

agent_df = spark.table(
"workspace.insurance_silver.agent_master"
)

policy_type_df = spark.table(
"workspace.insurance_silver.policy_type_master"
)

from pyspark.sql.functions import *

In [0]:
premium_summary = (

payment_df

.groupBy(
    "payment_date"
)

.agg(

    sum("payment_amount").alias(
        "total_premium_collected"
    ),

    countDistinct(
        "payment_id"
    ).alias(
        "total_payments"
    ),

    countDistinct(
        "policy_id"
    ).alias(
        "total_policies"
    ),

    avg("payment_amount").alias(
        "average_payment_amount"
    )

)

)

premium_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "workspace.insurance_gold.premium_summary"
)

In [0]:
policy_performance = (

payment_df

.groupBy(

    "policy_id",
    "policy_type",
    "policy_status",
    "payment_frequency"

)

.agg(

    sum("payment_amount").alias(
        "total_premium_collected"
    ),

    countDistinct(
        "payment_id"
    ).alias(
        "total_payments"
    ),

    max("annual_premium").alias(
        "annual_premium"
    ),

    max("sum_insured").alias(
        "sum_insured"
    ),

    max("outstanding_premium").alias(
        "outstanding_premium"
    )

)

)

policy_performance.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "workspace.insurance_gold.policy_performance"
)

In [0]:
customer_performance = (

payment_df

.groupBy(

    "customer_id",
    "customer_name",
    "gender",
    "customer_segment",
    "city",
    "state"

)

.agg(

    sum("payment_amount").alias(
        "total_premium_paid"
    ),

    countDistinct(
        "payment_id"
    ).alias(
        "total_payments"
    ),

    countDistinct(
        "policy_id"
    ).alias(
        "total_policies"
    ),

    avg("payment_amount").alias(
        "average_payment"
    )

)

)

customer_performance.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "workspace.insurance_gold.customer_performance"
)

In [0]:
agent_performance = (

payment_df

.groupBy(

    "agent_id",
    "agent_name",
    "branch_city",
    "agent_status"

)

.agg(

    sum("payment_amount").alias(
        "total_premium_collected"
    ),

    countDistinct(
        "payment_id"
    ).alias(
        "total_payments"
    ),

    countDistinct(
        "policy_id"
    ).alias(
        "total_policies"
    ),

    countDistinct(
        "customer_id"
    ).alias(
        "total_customers"
    )

)

)

agent_performance.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "workspace.insurance_gold.agent_performance"
)

In [0]:
policy_health = (

policy_df

.withColumn(

    "policy_health",

    when(

        col("policy_status") == "expired",

        "expired"

    )

    .when(

        col("policy_status") == "lapsed",

        "lapsed"

    )

    .when(

        col("outstanding_premium") > 0,

        "premium_outstanding"

    )

    .otherwise(

        "active"

    )

)

)

policy_health.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "workspace.insurance_gold.policy_health"
)

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_premium_summary_daily AS

SELECT

    payment_date,

    SUM(payment_amount) AS total_premium_collected,

    COUNT(DISTINCT payment_id) AS total_payments,

    COUNT(DISTINCT policy_id) AS total_policies

FROM workspace.insurance_silver.insurance_payments

GROUP BY
    payment_date;

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_premium_summary_monthly AS

SELECT

    YEAR(payment_date) AS year,

    MONTH(payment_date) AS month,

    SUM(payment_amount) AS total_premium_collected,

    COUNT(DISTINCT payment_id) AS total_payments,

    COUNT(DISTINCT policy_id) AS total_policies

FROM workspace.insurance_silver.insurance_payments

GROUP BY

    YEAR(payment_date),

    MONTH(payment_date);

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_agent_performance AS

SELECT

    agent_name,

    branch_city,

    agent_status,

    SUM(payment_amount) AS total_premium_collected,

    COUNT(DISTINCT payment_id) AS total_payments,

    COUNT(DISTINCT policy_id) AS total_policies,

    COUNT(DISTINCT customer_id) AS total_customers

FROM workspace.insurance_silver.insurance_payments

GROUP BY

    agent_name,

    branch_city,

    agent_status;

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_policy_performance AS

SELECT

    policy_type,

    policy_status,

    payment_frequency,

    SUM(payment_amount) AS total_premium_collected,

    COUNT(DISTINCT payment_id) AS total_payments,

    COUNT(DISTINCT policy_id) AS total_policies,

    SUM(outstanding_premium) AS total_outstanding_premium

FROM workspace.insurance_silver.insurance_payments

GROUP BY

    policy_type,

    policy_status,

    payment_frequency;

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_customer_performance AS

SELECT

    customer_segment,

    state,

    COUNT(DISTINCT customer_id) AS total_customers,

    COUNT(DISTINCT policy_id) AS total_policies,

    SUM(payment_amount) AS total_premium_paid,

    AVG(payment_amount) AS average_payment_amount

FROM workspace.insurance_silver.insurance_payments

GROUP BY

    customer_segment,

    state;

In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.insurance_gold.vw_policy_health AS

SELECT

    policy_status,

    COUNT(DISTINCT policy_id) AS total_policies,

    SUM(annual_premium) AS total_annual_premium,

    SUM(outstanding_premium) AS total_outstanding_premium,

    SUM(sum_insured) AS total_sum_insured

FROM workspace.insurance_silver.policy_master

GROUP BY

    policy_status;